# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlesakinnurun/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.** I picked it because the decision is concrete and the output is something a team can actually use: a ranked queue of pages to review first. The starter slice already has the shape of the problem — 30,000 anonymized pages, observed 90-day search and engagement signals, and a `trend_direction` bucket — so I can frame the question now and move to the warehouse release later for a future-window label.

It is provisional on purpose: if the signal audit shows the trend buckets are mostly noise, I will switch to Lane 4 (CTR / engagement) or the growth/recovery freestyle direction. The decision — who reviews what first, with limited capacity — stays the same.

## 2. The question: decision, action, cost of a wrong call

**The question.** Given a fixed weekly review capacity, which pages should a content editor open first for refresh, expansion, protection, pruning, or monitoring?

**The decision.** Not "predict decline" — the decision is *ordering the review queue*. The team sees a ranked list and works down it until the week's capacity is spent.

**Who acts.** A small content team with capacity for about **50 page reviews per week**. For each page, the reviewer chooses one action: refresh/update, expand, protect, prune/consolidate, or monitor.

**What a wrong call costs.**

- **False positive** (ranked high, nothing wrong): one of the ~50 weekly review slots is wasted — an editor-hour plus the opportunity cost of a real problem page that went unreviewed.
- **False negative** (a real decline ranked low): the decline continues for another cycle, so lost impressions and clicks compound.

Because capacity is fixed and the queue is the product, both errors are judged at the **top** of the list. The primary metric is **precision@50**; recall and average precision are secondary checks, not the headline.

**Why data/ML helps.** Each page carries a dozen noisy signals — impressions, clicks, position, CTR, freshness, engagement, age — and the useful pattern is a tangle of interactions, not one threshold. A plain rule covers the obvious cases, but the thresholds are policy choices. A learned ranking can weigh the signals together, and the later assignments will test whether it actually beats a transparent rule.

## 3. Quick look at the data (2-3 real numbers)

From the starter CSV (`data/raw/content_refresh_anonymized.csv`):

1. **30,000 pages, one row per page**, spread across **32 anonymized clients**.
2. **54.2%** of pages are labeled `down`; among the **16,726 pages with at least 500 impressions in 90 days**, **59.6%** are `down`. Decline is the majority class, not a rare event — so the job is ordering a large pool, not spotting an anomaly.
3. **Median 90-day impressions is 731.** At 50 reviews/week (about 2,600 per year), the 16,726 visible pages would take roughly **6.4 years** of capacity to review. Something has to decide what comes first.

Caveat: `trend_direction` is calculated from the current window, so it is a **proxy label**, not a future outcome. ML-04 replaces it with a prior-window → future-window label built from the warehouse release.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_pages = len(df)
n_clients = df["client_id"].nunique()
down_share = (df["trend_direction"] == "down").mean()

visible = df[df["impressions_90d"] >= 500]
visible_down_share = (visible["trend_direction"] == "down").mean()
median_impressions = df["impressions_90d"].median()

weekly_capacity = 50
annual_capacity = weekly_capacity * 52
years_to_cover = len(visible) / annual_capacity

print(f"pages: {n_pages:,} | clients: {n_clients}")
print(f"share labeled down (all pages): {down_share:.1%}")
print(
    f"pages with >=500 impressions: {len(visible):,} "
    f"| share labeled down: {visible_down_share:.1%}"
)
print(f"median 90-day impressions: {median_impressions:,.0f}")
print(
    f"visible pages at {weekly_capacity}/week "
    f"({annual_capacity:,}/year): {years_to_cover:.1f} years to review"
)

pages: 30,000 | clients: 32
share labeled down (all pages): 54.2%
pages with >=500 impressions: 16,726 | share labeled down: 59.6%
median 90-day impressions: 731
visible pages at 50/week (2,600/year): 6.4 years to review


## 4. Careful words: what I can and can't claim

**I can claim:**

- **Observed** associations in this anonymized slice — for example, "in the starter slice, pages with signal X had a higher observed share of `down` labels."
- **Directional, decision-support** results — for example, "on held-out clients, the queue put pages with these observed signals first and beat the baseline rule at precision@50."
- **Measured** performance under a validation design I state plainly (client holdout, time-aware split, or both).

**I cannot claim:**

- Causal proof that refreshing a page **causes** recovery — that needs an experiment or a causal design.
- That these signals are Google ranking factors.
- Anything about AI citations, AI rankings, or AI search visibility.
- Anything identifying a client, domain, URL, or raw query.
- That a high score guarantees a page is "bad" or that a refresh will pay off.

Every claim in this notebook and the final paper will use **observed / measured / directional / decision-support** language.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.